In [ ]:
import pandas as pd
import time

col_names = ['Name', 'G','GS', 'QBrec', 'Cmp','Att','Cmp%','Yds','TD','TD%','Int','Int%','1D','Succ%','Lng','Y/A','AY/A','Y/C','Y/G','Rate','Sk','Yds.1','Sk%','NY/A','ANY/A','4QC','GWD','AV','HOF']
stat_df = pd.DataFrame(columns=col_names)

# Load passing_leaders_filtered.csv into a DataFrame
passing_leaders_df = pd.read_csv('passing_leaders_filtered.csv')

special = {"Joe Montana": "https://www.pro-football-reference.com/players/M/MontJo01.htm", 
           "Alex Smith": "https://www.pro-football-reference.com/players/S/SmitAl03.htm", 
           "Danny White": "https://www.pro-football-reference.com/players/W/WhitDa01.htm",
           "Steve Deberg": "https://www.pro-football-reference.com/players/D/DebeSt00.htm",
           "Charley Johnson": "https://www.pro-football-reference.com/players/J/JohnCh02.htm"}

starting_name = "Peyton Manning"
start = False

only_one = False

for leader_row in passing_leaders_df.itertuples():
    # Create the URL for the player's PFR page
    player = leader_row.Player

    if player == starting_name:
        start = True

    if not start:
        continue
    
    # Split player name into first and last name
    first_name, last_name = player.split(" ", 1)
    last_name = last_name.replace(" ", "").replace("'", "")

    # Deal with players who have alt url
    if player in special:
        url = special[player]
    else:
        url = f'https://www.pro-football-reference.com/players/{last_name[0]}/{last_name[:4]}{first_name[0:2]}00.htm'

    print(url)

    # Use pandas to read HTML tables from the URL. It returns a list of DataFrames.
    try:
        df = pd.read_html(url)[0]
        # Find row in DataFrame where "Season" column contains "Yrs" in string, the first will be the career totals
        
        row = df[df['Season'].astype(str).str.contains('Yrs', na=False)].iloc[0]
        row["HOF"] = leader_row.HOF
        row["Name"] = leader_row.Player

        print(row)


        # Placeholder value for missing entries in the new row
        placeholder_value = None

        reindexed_series = row.reindex(stat_df.columns, fill_value=placeholder_value)
        row = reindexed_series.to_frame().T

        # Concatenate the original DataFrame and the new row DataFrame
        # Use ignore_index=True to reset the index of the resulting DataFrame
        stat_df = pd.concat([stat_df, row], ignore_index=True)

        print(f"Retrieved data for {player}\n")

    except Exception as e:
        print(f"Could not retrieve data for {player}: {e}\n")

    # May not exceed 10 requests per minute
    time.sleep(15)

    # save stat_df to csv
    stat_df.to_csv('career_stats.csv', index=False)

    if only_one:
        break






https://www.pro-football-reference.com/players/M/MannPe00.htm
Could not retrieve data for Peyton Manning: HTTP Error 403: Forbidden

https://www.pro-football-reference.com/players/F/FavrBr00.htm
Could not retrieve data for Brett Favre: HTTP Error 403: Forbidden

